# M1 Notebook 19 — Maximum Likelihood and Bayesian Inference

**Notebook ID:** M1_N19  
**Status:** Runnable first edition  
**Random seed:** 42


## Learning objectives

1. Define likelihood and log-likelihood.
2. Derive Bernoulli, Normal, and Poisson MLEs.
3. Compare MLE, MAP, and posterior mean.
4. Perform Beta–Bernoulli and Gamma–Poisson updating.
5. Construct Bayesian credible intervals.
6. Apply posterior uncertainty to Decision Intelligence.


In [ ]:
from srai_math.utils import environment_info,set_seed
from srai_math.statistics import (
    bernoulli_log_likelihood,bernoulli_mle,beta_bernoulli_posterior,
    beta_credible_interval,beta_posterior_mean,gamma_poisson_posterior,
    gamma_posterior_mean,map_beta_bernoulli,normal_log_likelihood,
    normal_mle,poisson_log_likelihood,poisson_mle,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
set_seed(42)
environment_info()


## Likelihood and log-likelihood

\[
L(\theta;x)=p(x\mid\theta),\qquad
\ell(\theta;x)=\log L(\theta;x).
\]

Maximum likelihood selects the parameter maximizing this function of \(\theta\).


In [ ]:
binary=np.array([1,1,0,1,1,0,1,1,1,0])
p_hat=bernoulli_mle(binary)
grid=np.linspace(.01,.99,500)
ll=np.array([bernoulli_log_likelihood(p,binary) for p in grid])
assert np.isclose(grid[np.argmax(ll)],p_hat,atol=.01)
p_hat


In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
ax.plot(grid,ll)
ax.axvline(p_hat,linestyle="--",label=f"MLE={p_hat:.2f}")
ax.set_xlabel("p"); ax.set_ylabel("Log-likelihood")
ax.set_title("Bernoulli Log-Likelihood"); ax.legend()
plt.show()


For Bernoulli observations,

\[
\hat p_{\mathrm{MLE}}=\bar x.
\]


In [ ]:
normal_data=np.array([9.8,10.2,10.5,9.9,10.1,10.4,9.7])
mu_hat,sigma_hat=normal_mle(normal_data)
{"mean_MLE":mu_hat,"standard_deviation_MLE":sigma_hat}


For Normal data,

\[
\hat\mu=\bar x,\qquad
\hat\sigma^2=\frac{1}{n}\sum_i(x_i-\bar x)^2.
\]


In [ ]:
counts=np.array([2,4,3,5,1,4,2,3,6,2])
rate_hat=poisson_mle(counts)
rates=np.linspace(.2,7,400)
pll=np.array([poisson_log_likelihood(r,counts) for r in rates])
assert np.isclose(rates[np.argmax(pll)],rate_hat,atol=.03)
rate_hat


## Bayesian inference

\[
p(\theta\mid x)
\propto
p(x\mid\theta)p(\theta).
\]


In [ ]:
successes=int(binary.sum()); failures=int(binary.size-successes)
a0,b0=2.,2.
a1,b1=beta_bernoulli_posterior(a0,b0,successes,failures)
summary={
    "MLE":p_hat,
    "posterior_mean":beta_posterior_mean(a1,b1),
    "MAP":map_beta_bernoulli(a0,b0,successes,failures),
    "credible_interval":beta_credible_interval(a1,b1),
}
summary


In [ ]:
p=np.linspace(.001,.999,500)
prior=stats.beta.pdf(p,a0,b0)
posterior=stats.beta.pdf(p,a1,b1)
scaled_likelihood=np.exp(np.array([bernoulli_log_likelihood(v,binary) for v in p])-ll.max())
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(p,prior,label="Prior")
ax.plot(p,scaled_likelihood,label="Scaled likelihood")
ax.plot(p,posterior,label="Posterior")
ax.set_xlabel("Probability"); ax.set_ylabel("Relative density")
ax.set_title("Beta–Bernoulli Updating"); ax.legend()
plt.show()


## MLE, MAP, and posterior mean

MLE uses the likelihood only. MAP maximizes the posterior. The posterior mean averages over posterior uncertainty.


In [ ]:
small=np.array([1,1,0])
s=int(small.sum()); f=int(small.size-s)
a,b=beta_bernoulli_posterior(20,20,s,f)
pd.Series({
    "MLE":bernoulli_mle(small),
    "MAP":map_beta_bernoulli(20,20,s,f),
    "Posterior mean":beta_posterior_mean(a,b),
})


## Gamma–Poisson updating

In [ ]:
shape,rate=gamma_poisson_posterior(
    shape=2.,rate=1.,total_count=int(counts.sum()),exposure=float(counts.size)
)
{"Poisson_MLE":rate_hat,"posterior_mean_rate":gamma_posterior_mean(shape,rate)}


## Credible versus confidence intervals

A Bayesian credible interval contains a specified amount of posterior probability. A frequentist confidence procedure has long-run coverage under repeated sampling.


## Decision Intelligence case — Service reliability

In [ ]:
a,b=beta_bernoulli_posterior(8,2,92,8)
service={
    "MLE_reliability":.92,
    "posterior_mean":beta_posterior_mean(a,b),
    "95_percent_credible_interval":beta_credible_interval(a,b),
    "posterior_probability_below_0_90":stats.beta.cdf(.90,a,b),
}
service


The posterior quantifies uncertainty under the model. Management action also depends on monitoring quality, failure severity, costs, and operational context.


## Engineering notes

- Use log-likelihoods to avoid underflow.
- Flat likelihoods indicate weak identifiability.
- Priors must be documented and stress-tested.
- Conjugate models are convenient but may oversimplify reality.
- Posterior inference is not itself a decision rule.


## Exercises

### Level A
Distinguish likelihood, prior, posterior, MLE, and MAP.

### Level B
Derive the Bernoulli and Poisson MLEs.

### Level C
Implement grid-based Bayesian inference for a non-conjugate model.

### Capstone
Estimate service reliability with likelihood and Bayesian methods, compare estimates and uncertainty, and define a separate decision rule.


## Key insight

Maximum likelihood identifies parameters most compatible with observed data. Bayesian inference combines evidence with prior information and preserves uncertainty through the posterior distribution.
